Preparo dos dados e verificação da configuração

Notebook desenvolvido para solução de trabalho prático usando Spark
Em alguns casos de solicitação de máximos valores, a seleção foi feita com 3 valores, por exemplo, apenas para título de conferência e treinamento

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("imdb_spark").getOrCreate()

df_titles = spark.read.csv("title_basics.tsv", header=True, inferSchema=True, sep="\t", nullValue="\\N")
df_ratings = spark.read.csv("title_ratings.tsv", header=True, inferSchema=True, sep="\t", nullValue="\\N")

df = df_titles.join(df_ratings, on="tconst", how="inner")

df.printSchema()
df.show(5)

root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- averageRating: double (nullable = true)
 |-- numVotes: integer (nullable = true)



+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+-----------------+-------------+--------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|           genres|averageRating|numVotes|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+-----------------+-------------+--------+
|tt0000002|    short|Le clown et ses c...|Le clown et ses c...|      0|     1892|   NULL|             5|  Animation,Short|          6.0|     233|
|tt0000004|    short|         Un bon bock|         Un bon bock|      0|     1892|   NULL|            12|  Animation,Short|          6.1|     152|
|tt0000008|    short|Edison Kinetoscop...|Edison Kinetoscop...|      0|     1894|   NULL|             1|Documentary,Short|          5.5|    1965|
|tt0000015|    short| Autour d'une cabine| Autour d'une cabine|      0|     1894|   NULL|             2|  Animation,Short|  

Quantos filmes (incluindo os da televisão) foram lançados no ano de 2015?

In [26]:
# conferindo os possíveis valores de titleType primeiro
df_titles.select("titleType").distinct().show()

+------------+
|   titleType|
+------------+
|    tvSeries|
|tvMiniSeries|
|     tvMovie|
|   tvEpisode|
|       movie|
|   tvSpecial|
|       video|
|   videoGame|
|     tvShort|
|       short|
|     tvPilot|
| radioSeries|
|radioEpisode|
+------------+



In [27]:
df_titles.filter(
    (F.col("startYear") == 2015) &
    (F.col("titleType").isin("movie", "tvMovie"))
).count()

19987

Qual o gênero de títulos mais frequente?

In [28]:
# após o df.show(5) do início, vemos que existem mais de um gênero em alguns casos na coluna, separados por vírgula

df_titles.withColumn("genre", F.explode(F.split(F.col("genres"), ","))) \
    .groupBy("genre") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+-----------+-------+
|      genre|  count|
+-----------+-------+
|      Drama|2247995|
|     Comedy|1653725|
|      Short|1021850|
|  Talk-Show| 900198|
|Documentary| 764885|
|    Romance| 724729|
|     Family| 571470|
|       News| 524662|
| Reality-TV| 423455|
|  Animation| 406284|
|      Music| 394008|
|      Crime| 351447|
|     Action| 334580|
|  Adventure| 324325|
|  Game-Show| 252533|
|      Adult| 242704|
|      Sport| 178594|
|    Fantasy| 174119|
|    Mystery| 162448|
|     Horror| 146400|
+-----------+-------+
only showing top 20 rows


Qual o gênero com a melhor nota média de títulos?

In [29]:
df.withColumn("genre", F.explode(F.split(F.col("genres"), ","))) \
    .groupBy("genre") \
    .agg(F.avg("averageRating").alias("genreAverageRating")) \
    .orderBy(F.desc("genreAverageRating")) \
    .show()

+-----------+------------------+
|      genre|genreAverageRating|
+-----------+------------------+
|    History| 7.353780102645095|
|Documentary| 7.240198535554613|
|  Biography| 7.175531914893609|
|    Mystery| 7.170086406897955|
|      Crime|  7.15984286848599|
|  Adventure| 7.107629703351801|
|    Fantasy|  7.09514565084539|
|  Animation| 7.089381171483268|
|    Western| 7.080683426568721|
|     Family| 7.070054926034514|
|      Drama| 7.040979155040113|
|        War|  7.00911513441492|
|     Action| 7.007098138747912|
|      Sport| 6.966792418526416|
|     Comedy| 6.960016550918304|
|      Music| 6.927469624015725|
| Reality-TV| 6.892611170895986|
|  Game-Show| 6.876828101904181|
|    Romance|6.8640161647039815|
|      Short| 6.791292438368553|
+-----------+------------------+
only showing top 20 rows


Qual o vídeo game do gênero aventura mais bem avaliado em 2020?

In [30]:
'''Para a resposta, foi considerado "mais bem avaliado" o de maior nota. Os outros valores trazidos 
e a lista dos 10 primeiros serviriam para avaliar outliers, se fosse o caso
'''
df.filter(
    (F.col("titleType") == "videoGame") &
    (F.col("startYear") == 2020) &
    (F.col("genres").contains("Adventure"))
) \
    .orderBy(F.desc("averageRating")) \
    .select("primaryTitle", "averageRating", "numVotes") \
    .show(10, truncate=False)

+----------------------------------------+-------------+--------+
|primaryTitle                            |averageRating|numVotes|
+----------------------------------------+-------------+--------+
|Half-Life: Alyx                         |9.5          |506     |
|Ghost of Tsushima                       |9.3          |5270    |
|Omori                                   |9.2          |79      |
|Ori and the Will of the Wisps           |9.1          |724     |
|Final Fantasy VII Remake                |9.1          |2749    |
|There Is No Game: Wrong Dimension       |8.9          |39      |
|Mega Man Zero/ZX Legacy Collection      |8.9          |11      |
|Xenoblade Chronicles: Definitive Edition|8.8          |128     |
|Yakuza: Like a Dragon                   |8.8          |279     |
|Demon's Souls                           |8.8          |439     |
+----------------------------------------+-------------+--------+
only showing top 10 rows


Quantos títulos diferentes existem?

In [31]:
df_titles.select('primaryTitle').distinct().count()

3931670

Qual a duração média dos títulos com conteúdo adulto?

In [32]:
df_titles.filter(F.col("isAdult") == 1) \
    .describe("runtimeMinutes") \
    .show()

+-------+-----------------+
|summary|   runtimeMinutes|
+-------+-----------------+
|  count|            91464|
|   mean|92.79938555059914|
| stddev|57.18982244754778|
|    min|                1|
|    max|               99|
+-------+-----------------+



Quantos títulos têm o título atual (“primary”) diferente do título original?

In [33]:
df_titles.filter(F.col("primaryTitle") != F.col("originalTitle")).count()

125056

Qual o título que tem o nome mais longo?

In [34]:
df_titles.orderBy(F.length(F.col("primaryTitle")).desc()) \
    .select("tconst", "primaryTitle") \
    .show(3, truncate=False)

+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tconst    |primaryTitle                                                                                                                                                                                                                                                                                                                                                                                                                       |
+----------+--------------------------------------------------------------------------------------------------------------------------

Qual título tem a maior quantidade de votos? 

In [35]:
# a questão pedia especificamente para se usar o describe na solução, por isso a estratégia abaixo
df.describe("numVotes").show()

+-------+------------------+
|summary|          numVotes|
+-------+------------------+
|  count|           1182639|
|   mean| 973.0778656885153|
| stddev|16275.709043258415|
|    min|                 5|
|    max|           2449517|
+-------+------------------+



In [36]:
df.filter(F.col("numVotes") == 2449517).show()

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+------+-------------+--------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|genres|averageRating|numVotes|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+------+-------------+--------+
|tt0111161|    movie|The Shawshank Red...|The Shawshank Red...|      0|     1994|   NULL|           142| Drama|          9.3| 2449517|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+------+-------------+--------+



In [37]:
# Uma alternativa mais robusta seria fazer o que já aplicamos em outras questões acima
# Ou seja, já ordenar pelo máximo valor
df.orderBy(F.desc("numVotes")) \
    .select("tconst", "primaryTitle", "numVotes") \
    .show(3, truncate=False)

+---------+------------------------+--------+
|tconst   |primaryTitle            |numVotes|
+---------+------------------------+--------+
|tt0111161|The Shawshank Redemption|2449517 |
|tt0468569|The Dark Knight         |2405191 |
|tt1375666|Inception               |2157649 |
+---------+------------------------+--------+
only showing top 3 rows


Qual é a menor nota média de um título? 

In [38]:
# A questão pedia o uso de describe()
df.describe("averageRating").show()

+-------+------------------+
|summary|     averageRating|
+-------+------------------+
|  count|           1182639|
|   mean| 6.917028357766055|
| stddev|1.3974964575775854|
|    min|               1.0|
|    max|              10.0|
+-------+------------------+



In [39]:
# Se a intenção fosse armazenar numa variável, mesmo que usando describe() poderíamos seguir com
stats = df.describe("averageRating")
menor_nota = float(stats.filter(F.col("summary") == "min").collect()[0]["averageRating"])
menor_nota

1.0